Multi-armed Bandits Hyper-parameter tuning

In [2]:
import torch
import math, os
import torch.nn as nn
import torch.optim as optim
from dataclasses import dataclass
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, TensorDataset

Config class

In [3]:
@dataclass
class ParameterConfig:
    learning_rate: float
    batch_size: int
    hidden_layers: list[int]

In [8]:
CONFIGURATIONS: list[ParameterConfig] = [
    ParameterConfig(learning_rate=0.1, batch_size=64, hidden_layers=[128]),
    ParameterConfig(learning_rate=0.01, batch_size=128, hidden_layers=[128, 64]),
    ParameterConfig(learning_rate=0.001, batch_size=32, hidden_layers=[64]),
]

Params

In [9]:
input_dim = 28 * 28 # MNIST input dimension
output_dim = 10 # MNIST output class
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Persistent Storage Arrays

In [11]:
active_models = [None] * len(CONFIGURATIONS)
active_optimizers = [None] * len(CONFIGURATIONS)

Dynamically build the Trainable Network

In [5]:
def build_neural_network(config: ParameterConfig) -> nn.Sequential:
    layers = nn.Sequential()
    init_dim = input_dim

    for hidden_dim in config.hidden_layers:
        layers.append(nn.Linear(init_dim, hidden_dim))
        layers.append(nn.ReLU())
        init_dim = hidden_dim
    
    layers.append(nn.Linear(init_dim, output_dim))
    return layers

Testing the function

In [7]:
testmodel = build_neural_network(CONFIGURATIONS[0])
testmodel

Sequential(
  (0): Linear(in_features=784, out_features=128, bias=True)
  (1): ReLU()
  (2): Linear(in_features=128, out_features=10, bias=True)
)

Preparing the dataset

In [10]:
transform_pipeline = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_set = datasets.MNIST(root='/home/bukunmi/ml-journey/datasets/data', train=True, download=True, transform=transform_pipeline)
test_set = datasets.MNIST(root='/home/bukunmi/ml-journey/datasets/data', train=False, download=True, transform=transform_pipeline)

Bandit Selection Engine